# Phoenix Event Display Explore

In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import uproot
import glob
import awkward as ak
import itertools
import yaml
import os
import sys
from tqdm import tqdm
from pathlib import Path
import atlasify as atl
atl.ATLAS = "ColliderML"

sys.path.append("../")
from utils import load_root_file, load_hepmc_event
from edm4hep_utils import build_calo_df, build_particle_df, build_tracker_df, load_edm4hep_file

## Roadmap

1. Load in edm4hep file
2. Copy in edm4hep2json script
3. Convert script from c++ to python
4. Run script
5. Load json file into Phoenix
6. Visualize events


In [32]:
edm4hep_file = "/global/cfs/cdirs/m3443/usr/dtmurnane/Side_Work/ACTS/outputs/pda_testing16/edm4hep.root"
# all_events = load_edm4hep_file(edm4hep_file)
event = load_edm4hep_file(edm4hep_file, event_num=0)

In [33]:
event

{'tracker_df':                  cellID      EDep       time  pathLength     quality  \
 0       241894806257926  0.001156   0.901782    0.182693           0   
 1       239760123625990  0.001629   0.951476    0.196319           0   
 2        33671218203142  0.000079   0.295579    0.156231           0   
 3       246633849754390  0.000065   0.502378    0.156047           0   
 4         7419891621414  0.000109   0.785662    0.168867           0   
 ...                 ...       ...        ...         ...         ...   
 209021   16767552426254  0.000014  14.811994    0.001206  1073741824   
 209022    1155346297118  0.000014  17.057121    0.002514  1073741824   
 209023     137440174350  0.000007  20.332094    0.000900  1073741824   
 209024   16110422437902  0.000217  15.784687    0.532923  1073741824   
 209025     300648759582  0.000004  22.262377    0.000361  1073741824   
 
                   x           y            z        px        py        pz  \
 0         26.553848   17.972

In [34]:
tracker_df = event["tracker_df"]

In [35]:
tracker_df['color'] = np.where(
                tracker_df['detector'].str.contains('Pixel'),
                "#FF0000",  # Red for pixel
                np.where(
                    tracker_df['detector'].str.contains('Strip'),
                    "#00FF00",  # Green for strips
                    "#FF0000"   # Default red
                )
            )

In [36]:
tracker_df

,cellID,EDep,time,pathLength,quality,x,y,z,px,py,pz,particle_id,r,phi,pt,detector,color
0,241894806257926,0.001156,0.901782,0.182693,0,26.553848,17.972034,-500.026505,0.093749,0.052184,-0.114956,77345,32.064012,0.594995,0.107295,PixelBarrelReadout,#FF0000
1,239760123625990,0.001629,0.951476,0.196319,0,27.915682,18.713187,-501.638857,0.079670,0.045392,-0.099218,77345,33.607568,0.590546,0.091694,PixelBarrelReadout,#FF0000
2,33671218203142,0.000079,0.295579,0.156231,0,-26.043205,-20.182690,-446.778574,-0.094212,-0.060381,0.069811,77335,32.948285,-2.482302,0.111900,PixelBarrelReadout,#FF0000
3,246633849754390,0.000065,0.502378,0.156047,0,-57.528082,-36.911998,-424.150507,-0.101101,-0.041742,0.069870,77335,68.351853,-2.571121,0.109379,PixelBarrelReadout,#FF0000
4,7419891621414,0.000109,0.785662,0.168867,0,-104.302509,-48.428373,-393.423990,-0.106607,-0.011504,0.068167,77335,114.997047,-2.706905,0.107226,PixelBarrelReadout,#FF0000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
209021,16767552426254,0.000014,14.811994,0.001206,1073741824,-1041.277048,-164.817290,1304.375443,0.000017,0.000024,0.000053,3004119,1054.240309,-2.984611,0.000029,LongStripEndcapReadout,#00FF00
209022,1155346297118,0.000014,17.057121,0.002514,1073741824,-1033.752384,178.003525,1604.385709,-0.000016,0.000037,0.000043,1987,1048.965798,2.971073,0.000041,LongStripEndcapReadout,#00FF00
209023,137440174350,0.000007,20.332094,0.000900,1073741824,698.340773,-691.428360,1320.440813,0.000005,-0.000043,-0.000001,1987,982.727334,-0.780424,0.000044,LongStripEndcapReadout,#00FF00
209024,16110422437902,0.000217,15.784687,0.532923,1073741824,-694.784582,-344.854033,1274.377225,-0.000147,0.000064,0.000112,3004287,775.660956,-2.680872,0.000161,LongStripEndcapReadout,#00FF00


In [37]:
tracker_df.assign(
                    type="Point",
                    pos=lambda df: df[['x', 'y', 'z']].values.tolist(),
                    energy=lambda df: df['EDep']
                )

,cellID,EDep,time,pathLength,quality,x,y,z,px,py,pz,particle_id,r,phi,pt,detector,color,type,pos,energy
0,241894806257926,0.001156,0.901782,0.182693,0,26.553848,17.972034,-500.026505,0.093749,0.052184,-0.114956,77345,32.064012,0.594995,0.107295,PixelBarrelReadout,#FF0000,Point,"[26.553847914612007, 17.97203414366656, -500.0...",0.001156
1,239760123625990,0.001629,0.951476,0.196319,0,27.915682,18.713187,-501.638857,0.079670,0.045392,-0.099218,77345,33.607568,0.590546,0.091694,PixelBarrelReadout,#FF0000,Point,"[27.915681942147934, 18.713186716489105, -501....",0.001629
2,33671218203142,0.000079,0.295579,0.156231,0,-26.043205,-20.182690,-446.778574,-0.094212,-0.060381,0.069811,77335,32.948285,-2.482302,0.111900,PixelBarrelReadout,#FF0000,Point,"[-26.043205432293306, -20.182689569763753, -44...",0.000079
3,246633849754390,0.000065,0.502378,0.156047,0,-57.528082,-36.911998,-424.150507,-0.101101,-0.041742,0.069870,77335,68.351853,-2.571121,0.109379,PixelBarrelReadout,#FF0000,Point,"[-57.52808229621148, -36.91199799232626, -424....",0.000065
4,7419891621414,0.000109,0.785662,0.168867,0,-104.302509,-48.428373,-393.423990,-0.106607,-0.011504,0.068167,77335,114.997047,-2.706905,0.107226,PixelBarrelReadout,#FF0000,Point,"[-104.3025089806784, -48.42837337647842, -393....",0.000109
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
209021,16767552426254,0.000014,14.811994,0.001206,1073741824,-1041.277048,-164.817290,1304.375443,0.000017,0.000024,0.000053,3004119,1054.240309,-2.984611,0.000029,LongStripEndcapReadout,#00FF00,Point,"[-1041.2770475014895, -164.81729014690427, 130...",0.000014
209022,1155346297118,0.000014,17.057121,0.002514,1073741824,-1033.752384,178.003525,1604.385709,-0.000016,0.000037,0.000043,1987,1048.965798,2.971073,0.000041,LongStripEndcapReadout,#00FF00,Point,"[-1033.7523836424316, 178.00352499826374, 1604...",0.000014
209023,137440174350,0.000007,20.332094,0.000900,1073741824,698.340773,-691.428360,1320.440813,0.000005,-0.000043,-0.000001,1987,982.727334,-0.780424,0.000044,LongStripEndcapReadout,#00FF00,Point,"[698.3407728407393, -691.4283602050109, 1320.4...",0.000007
209024,16110422437902,0.000217,15.784687,0.532923,1073741824,-694.784582,-344.854033,1274.377225,-0.000147,0.000064,0.000112,3004287,775.660956,-2.680872,0.000161,LongStripEndcapReadout,#00FF00,Point,"[-694.7845818368769, -344.8540332827482, 1274....",0.000217


In [27]:
import json
import numpy as np
from pathlib import Path
from typing import List, Dict, Any, Optional
import uproot
from tqdm import tqdm

def process_collection(collection_name: str, collection_data: Any, collection_id: int) -> dict:
    """Convert a collection into the expected JSON format"""
    
    # Map our DataFrame names to EDM4hep collection types
    type_mapping = {
        "tracker_df": "TrackerHit",
        "ecal_hits_df": "CalorimeterHit",
        "ecal_contrib_df": "CaloHitContribution",
        "particles_df": "MCParticle",
        "parents_df": "MCParticleLink",
        "daughters_df": "MCParticleLink"
    }
    
    # Convert DataFrame to list of dictionaries
    collection_list = collection_data.to_dict(orient='records')
    
    # Handle numpy types for JSON serialization
    for item in collection_list:
        for key, value in item.items():
            if isinstance(value, np.integer):
                item[key] = int(value)
            elif isinstance(value, np.floating):
                item[key] = float(value)
            elif isinstance(value, np.ndarray):
                item[key] = value.tolist()
    
    collection_type = type_mapping.get(collection_name, collection_name)
    
    return {
        collection_type: {
            "type": f"edm4hep::{collection_type}Collection",
            "value": collection_list
        }
    }

def edm4hep_to_json(
    input_file: str,
    output_file: Optional[str] = None,
    requested_collections: Optional[List[str]] = None,
    event_num: int = None,
    verbose: bool = False
) -> None:
    """Convert EDM4hep file to JSON format matching C++ implementation"""
    
    # Handle output file path
    if output_file is None:
        output_path = Path(input_file)
        output_file = str(output_path.with_stem(output_path.stem.replace("edm4hep", "")).with_suffix('.edm4hep.json'))
    
    # Load events
    if verbose:
        print(f"Loading events from {input_file}")
    events_to_process = load_edm4hep_file(input_file, event_num=event_num)
    
    # Handle single event case
    if isinstance(events_to_process, dict):
        events_to_process = [events_to_process]
    
    if verbose:
        print(f"Processing {len(events_to_process)} events")
        print(events_to_process[0].keys())

    # Process events
    all_events_dict = {}
    for event_id, event_data in tqdm(enumerate(events_to_process)):
        event_dict = {
            "eventType": "edm4hep",
            "runNumber": 0,  # Add if available
            "eventNumber": event_id,
            "collections": {}
        }
        
        # Process collections
        collections = requested_collections or list(event_data.keys())
        for coll_name in collections:
            if coll_name in event_data:
                collection_dict = process_collection(
                    coll_name,
                    event_data[coll_name],
                    hash(coll_name) & 0xFFFFFFFF
                )
                event_dict["collections"].update(collection_dict)
        
        all_events_dict[f"Event{event_id}"] = event_dict
    
    # Write output
    if verbose:
        print(f"Writing output to {output_file}")
    with open(output_file, 'w') as f:
        json.dump(all_events_dict, f, indent=2)

In [50]:

def calculate_cell_color(energy: float, min_energy: float = 1e-3, max_energy: float = 1.0) -> str:
    """Convert energy to a color using HSL color space"""
    # Convert energy to lightness (inverted: higher energy = darker color)
    lightness = 80 - ((energy - min_energy) * 65) / (max_energy - min_energy)
    lightness = max(20, min(85, lightness))
    
    # Use a fixed hue (e.g., blue = 240) and saturation (90%)
    hue = 240
    saturation = 90
    
    # Convert HSL to hex color (implementation needed)
    return hsl_to_hex(hue, saturation, lightness)

def hsl_to_hex(h: float, s: float, l: float) -> str:
    """Convert HSL color values to hex color string"""
    l /= 100
    a = s * min(l, 1 - l) / 100

    def f(n):
        k = (n + h/30) % 12
        color = l - a * max(min(k - 3, 9 - k, 1), -1)
        return round(255 * color)

    return f"#{f(0):02x}{f(8):02x}{f(4):02x}"

In [51]:
def edm4hep_to_json(
    input_file: str,
    output_file: Optional[str] = None,
    requested_collections: Optional[List[str]] = None,
    event_num: int = None,
    verbose: bool = False
) -> None:
    """Convert EDM4hep file to Phoenix-compatible JSON format"""
    
    # Handle output file path
    if output_file is None:
        output_path = Path(input_file)
        output_file = str(output_path.with_suffix('').with_suffix('.json'))
    
    # Load events
    if verbose:
        print(f"Loading events from {input_file}")
    events_to_process = load_edm4hep_file(input_file, event_num=event_num)
    
    # Handle single event case
    if isinstance(events_to_process, dict):
        events_to_process = [events_to_process]
    
    # Process events
    all_events_dict = {}
    for event_id, event_data in enumerate(events_to_process):
        event_dict = {
            "event number": event_id,
            "run number": 0,
            "Hits": {},
            "Tracks": {},
            "CaloCells": {},
        }
        
        # Add tracker hits
        if "tracker_df" in event_data and "Hits" in requested_collections:
            tracker_df = event_data["tracker_df"]
            
            # Convert hits to Phoenix format
            tracker_hits = []
            for _, hit in tracker_df.iterrows():
                tracker_hits.append({
                    "type": "Point",
                    "pos": [
                        float(hit.x) * 0.1,  # Convert mm to cm
                        float(hit.y) * 0.1,
                        float(hit.z) * 0.1
                    ],
                    "color": "#FF0000"  # Default red color
                })
            
            event_dict["Hits"]["TrackerHits"] = tracker_hits
        
        # Add calorimeter hits
        if "ecal_hits_df" in event_data and "CaloCells" in requested_collections:
            calo_df = event_data["ecal_hits_df"]
            
            # Convert hits to Phoenix format
            calo_cells = []
            for _, cell in calo_df.iterrows():
                # Calculate eta and phi from x,y,z
                x = float(cell.x) * 0.1
                y = float(cell.y) * 0.1
                z = float(cell.z) * 0.1
                phi = float(cell.phi)
                eta = float(cell.eta)
                
                # Calculate color based on energy
                energy = float(cell.energy)
                color = calculate_cell_color(energy)  # Helper function needed
                
                calo_cells.append({
                    "energy": energy,
                    "phi": phi,
                    "eta": eta,
                    "color": color
                })
            
            event_dict["CaloCells"]["ECALCells"] = calo_cells
        
        all_events_dict[f"Event{event_id}"] = event_dict
    
    # Write output
    if verbose:
        print(f"Writing output to {output_file}")
    with open(output_file, 'w') as f:
        json.dump(all_events_dict, f, indent=2)


In [52]:
edm4hep_file = "/global/cfs/cdirs/m3443/usr/dtmurnane/Side_Work/ACTS/outputs/pda_testing16/edm4hep.root"
output_json_file = "/global/cfs/cdirs/m3443/usr/dtmurnane/Side_Work/ACTS/outputs/pda_testing16/one_event4.edm4hep.json"
# Export tracker information
edm4hep_to_json(edm4hep_file, output_json_file, event_num=0, verbose=True, requested_collections=["Hits"])


Loading events from /global/cfs/cdirs/m3443/usr/dtmurnane/Side_Work/ACTS/outputs/pda_testing16/edm4hep.root
Writing output to /global/cfs/cdirs/m3443/usr/dtmurnane/Side_Work/ACTS/outputs/pda_testing16/one_event4.edm4hep.json


In [53]:
# Load the json file to examine
import json
with open(output_json_file, 'r') as f:
    data = json.load(f)
data


{'Event0': {'event number': 0,
  'run number': 0,
  'Hits': {'TrackerHits': [{'type': 'Point',
     'pos': [2.6553847914612008, 1.797203414366656, -50.00265045691759],
     'color': '#FF0000'},
    {'type': 'Point',
     'pos': [2.7915681942147934, 1.8713186716489105, -50.163885731335384],
     'color': '#FF0000'},
    {'type': 'Point',
     'pos': [-2.604320543229331, -2.0182689569763754, -44.677857443052204],
     'color': '#FF0000'},
    {'type': 'Point',
     'pos': [-5.752808229621149, -3.6911997992326264, -42.41505074519093],
     'color': '#FF0000'},
    {'type': 'Point',
     'pos': [-10.43025089806784, -4.842837337647842, -39.34239904842034],
     'color': '#FF0000'},
    {'type': 'Point',
     'pos': [-16.358642351562526, -4.510380358922117, -35.33786925772743],
     'color': '#FF0000'},
    {'type': 'Point',
     'pos': [-14.078512566023528, 9.73840454355071, 2.7395392775937113],
     'color': '#FF0000'},
    {'type': 'Point',
     'pos': [-13.550891678749764, 10.31820751038